In [5]:
from typing import Optional

import conllu
import pandas as pd
from conllu import parse_tree
from tqdm import tqdm

In [2]:
sentences = []
with open("train_nlprepl-ud.conllu", "r", encoding="utf-8") as f:
    for sentence in conllu.parse_incr(f):
        sentences.append(sentence)

In [6]:
def preview_sentences(match_token_fn, limit=5):
    matched = 0
    for ix, sentence in tqdm(enumerate(sentences)):
        if limit is not None:
            if matched >= limit:
                break
        if match_token_fn(sentence.to_tree()):
            print(f"Conllu index={ix}, sentence= {sentence.metadata['text']}")
            matched += 1

In [11]:
# https://docs.google.com/document/d/1zJla81afT411Bro3sZeVdU0ayIomwK2w5Du06AcB8sw/edit?tab=t.0

def match_subj_verb_number_clause(tokenTree: conllu.TokenTree):
    token = tokenTree.token
    if token['upos'] == 'VERB' and token['deprel'] == 'root' and 'sg' in token['xpos']:
        for ch in tokenTree.children:
            if ch.token['upos'] == 'VERB' and ch.token['deprel'] == 'csubj' and 'inf' in ch.token['xpos'].split(":"):
                return True
    else:
        return False

preview_sentences(match_subj_verb_number_clause)

10388it [00:00, 50636.13it/s]

Conllu index=728, sentence= Zdawało się, że będą łowić sieciami księżyc odbity w wodzie, więc nie chcą głosami spłoszyć tej złotej ryby z okrągłym łbem.


41604it [00:00, 49726.85it/s]

Conllu index=31827, sentence= Przyszło jej na myśl, aby jeszcze raz przejrzeć dokumentację, którą Jeff zbierał do swojej pracy o zabytkach na Ukrainie i którą kiedyś Informator przesłał jej w paczce na dowód, że nie siedzi bezczynnie.


69360it [00:01, 51880.79it/s]


In [12]:
def match_subj_verb_gender_clause(tokenTree: conllu.TokenTree):
    token = tokenTree.token
    token_xpos = token['xpos'].split(":")
    if token['upos'] == 'VERB' and token['deprel'] == 'root' and 'n' in token_xpos and 'sg' in token_xpos:
        for ch in tokenTree.children:
            if ch.token['upos'] == 'VERB' and ch.token['deprel'] == 'csubj' and 'inf' in ch.token['xpos'].split(":"):
                return True
    else:
        return False

preview_sentences(match_subj_verb_gender_clause)

9765it [00:00, 43167.53it/s]

Conllu index=728, sentence= Zdawało się, że będą łowić sieciami księżyc odbity w wodzie, więc nie chcą głosami spłoszyć tej złotej ryby z okrągłym łbem.


42420it [00:00, 53001.14it/s]

Conllu index=31827, sentence= Przyszło jej na myśl, aby jeszcze raz przejrzeć dokumentację, którą Jeff zbierał do swojej pracy o zabytkach na Ukrainie i którą kiedyś Informator przesłał jej w paczce na dowód, że nie siedzi bezczynnie.


69360it [00:01, 52689.40it/s]


To zapytanie jest stricte podzbiorem poprzedniego zapytania.

Zapytanie 'subj_verb_number_infinitival' identyczne jak pierwsze

In [13]:
def match_subj_verb_number_pp_attractor(tokenTree: conllu.TokenTree):
    token = tokenTree.token
    token_xpos = token['xpos'].split(":")

    for number in ['sg', 'pl']:
        if token['upos'] == 'VERB' and token['deprel'] == 'root' and number in token_xpos:
            for ch in tokenTree.children:
                ch_token_xpos = ch.token['xpos'].split(":")
                if ch.token['upos'] == 'NOUN' and ch.token['deprel'] == 'nsubj' and number in ch_token_xpos:
                    for attractor in ch.children:
                        attractor.token['upos'] = 'NOUN'
                        attractor_xpos = attractor.token['xpos'].split(":")

                        other_number = 'pl' if number == 'sg' else 'sg'
                        if other_number in attractor_xpos:
                            for prep in attractor.children:
                                if prep.token['upos'] == 'ADP' and prep.token['deprel'] == 'case':
                                    return True

    return False

preview_sentences(match_subj_verb_number_pp_attractor)

3035it [00:00, 47354.57it/s]

Conllu index=1377, sentence= Nie wszystkie tomiki Broniewskiego sprzed roku 1939 dochowały się na moich półkach: Trzy salwy, Troska i pieśń, Krzyk ostateczny.
Conllu index=1843, sentence= Po krótkich bojach prawnych do tamtejszych sklepów trafiła ocenzurowana wersja okładki - bez swastyk.
Conllu index=2477, sentence= Krążki VideoCD oprócz ścieżki audio/wideo mogą zawierać m.in. menu, spis rozdziałów (podobnie jak w filmach na DVD), napisy oraz dodatkowe ścieżki dźwiękowe (stereo lub mono).
Conllu index=2671, sentence= Towarzysze z Moskwy przeprowadzali w Polsce szkolenia dla działaczy SdRP o zasadach działania partii w sferze gospodarczej - "pozwalających zabezpieczyć się przed próbami wywłaszczenia".
Conllu index=3034, sentence= 1 lutego 1982 r. - pojawiły się talony na benzynę (w zależności od pojemności samochodu - od 24 l do 45 l miesięcznie).
